# 🚀 Advanced Usage of Climatology Engine

This notebook demonstrates advanced usage of the Climatology Engine.

**What you will learn:**
- Working with Gridded Data
- Extreme Value Mode with GEV
- Advanced `config.yaml` settings
- Using checkpoint for resuming processing
- Memory management and block_size
- Saving in different formats (Zarr, NetCDF, CSV)
- Analyzing output with xarray
- Advanced mapping with Cartopy
- Performance optimization for large datasets

---

## 📐 Advanced config.yaml Settings

The `config.yaml` file controls all processing parameters.

### Key Parameters

| Parameter | Description | Default |
|-----------|-------------|---------|
| `block_size` | Number of points per block | 1000 |
| `n_points_max` | Maximum number of points | 40000 |
| `use_extreme_values` | Enable extreme value mode | false |
| `data_format` | Data format (station/gridded) | auto |
| `compression` | Compression algorithm | zstd |
| `cache_enabled` | Enable disk cache | true |

### Extreme Value Mode Settings

```yaml
window:
  days: 2
  use_extreme_values: true   # Enable GEV
```

### Gridded Data Settings

```yaml
data_format: "gridded"
lat_min: 25.0
lat_max: 40.0
lon_min: 44.0
lon_max: 64.0
```

---

In [ ]:
import sys
import os
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['font.size'] = 11
print('✅ Libraries loaded.')

In [ ]:
# ============================================================================
# 1. Load and inspect config.yaml
# ============================================================================

config_path = os.path.join(project_root, 'config.yaml')
if os.path.exists(config_path):
    with open(config_path, 'r', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    print("📋 Current settings:")
    print("=" * 60)
    
    print(f"block_size: {config.get('processing', {}).get('block_size', 'N/A')}")
    print(f"n_points_max: {config.get('processing', {}).get('n_points_max', 'N/A')}")
    print(f"use_extreme_values: {config.get('window', {}).get('use_extreme_values', 'N/A')}")
    print(f"data_format: {config.get('data_format', 'N/A')}")
    print(f"compression: {config.get('processing', {}).get('compression', 'N/A')}")
    print("=" * 60)
else:
    print("⚠️ config.yaml not found.")

In [ ]:
# ============================================================================
# 2. Generate Synthetic Gridded Data
# ============================================================================

def create_gridded_data(n_lat=20, n_lon=30, n_time=365):
    np.random.seed(42)
    lat = np.linspace(25, 40, n_lat)
    lon = np.linspace(44, 64, n_lon)
    
    time = np.arange(n_time)
    data = np.zeros((n_time, n_lat, n_lon))
    
    for i, t in enumerate(time):
        seasonal = 15 + 10 * np.sin(2 * np.pi * t / 365)
        spatial = np.outer(np.sin(lat * 0.1), np.cos(lon * 0.05))
        noise = np.random.normal(0, 2, (n_lat, n_lon))
        data[i, :, :] = seasonal + spatial * 5 + noise
    
    return lat, lon, data

lat, lon, gridded_data = create_gridded_data(n_lat=20, n_lon=30, n_time=365)

print("📊 Synthetic gridded data:")
print(f"   Shape: {gridded_data.shape}")
print(f"   Latitudes: {len(lat)} points ({lat.min():.1f} to {lat.max():.1f})")
print(f"   Longitudes: {len(lon)} points ({lon.min():.1f} to {lon.max():.1f})")
print(f"   Data range: [{gridded_data.min():.2f}, {gridded_data.max():.2f}]")

In [ ]:
# Save gridded data to NetCDF
netcdf_path = os.path.join(project_root, 'sample_data', 'gridded_sample.nc')
os.makedirs(os.path.dirname(netcdf_path), exist_ok=True)

ds_netcdf = xr.Dataset(
    data_vars={
        'tmean': (('time', 'lat', 'lon'), gridded_data),
    },
    coords={
        'time': np.arange(365),
        'lat': lat,
        'lon': lon,
    },
    attrs={'description': 'Synthetic gridded temperature data'}
)

ds_netcdf.to_netcdf(netcdf_path)
print(f"✅ Gridded data saved to {netcdf_path}")

In [ ]:
# Load gridded data from NetCDF
ds_loaded = xr.open_dataset(netcdf_path)
print("📂 Gridded data loaded:")
print(f"   Dimensions: {ds_loaded.dims}")
print(f"   Variables: {list(ds_loaded.data_vars)}")
print(f"   Coordinates: {list(ds_loaded.coords)}")

In [ ]:
# ============================================================================
# 3. Fitting on Gridded Data (converted to time series)
# ============================================================================

from core.engine.plugin_loader import load_plugins
from core.engine.distribution_plugin import DistributionPlugin

plugins = load_plugins()
distributions = {dist.name: dist for dist in plugins.values()}

lat_idx = 10
lon_idx = 15
point_data = gridded_data[:, lat_idx, lon_idx]

print(f"📊 Point data ({lat[lat_idx]:.2f}°, {lon[lon_idx]:.2f}°):")
print(f"   Mean: {np.mean(point_data):.2f}°C")
print(f"   Std: {np.std(point_data):.2f}°C")

normal_dist = distributions['Normal']
result = normal_dist.fit(point_data)

print("\n📈 Fit results:")
for key, value in result.items():
    if isinstance(value, float):
        print(f"   {key}: {value:.4f}")
    else:
        print(f"   {key}: {value}")

In [ ]:
# Plot time series of selected point
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(point_data, color='blue', alpha=0.7, linewidth=1.5)
ax.axhline(np.mean(point_data), color='red', linestyle='--', linewidth=2, label=f'Mean = {np.mean(point_data):.2f}°C')
ax.set_xlabel('Time (day)', fontsize=12)
ax.set_ylabel('Temperature (°C)', fontsize=12)
ax.set_title(f'Time Series at ({lat[lat_idx]:.2f}°, {lon[lon_idx]:.2f}°)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# 4. Extreme Value Mode with GEV
# ============================================================================

print("🔥 Extreme Value Mode:")
print("=" * 60)

seasonal_max = []
season_names = ['Winter', 'Spring', 'Summer', 'Autumn']
season_idx = [(0, 90), (91, 181), (182, 273), (274, 364)]

for start, end in season_idx:
    seasonal_max.append(np.max(point_data[start:end]))

print(f"📊 Seasonal maxima: {[f'{m:.2f}°C' for m in seasonal_max]}")

if 'GEV' in distributions:
    try:
        gev_dist = distributions['GEV']
        gev_result = gev_dist.fit(np.array(seasonal_max))
        print("\n📈 GEV fit results:")
        for key, value in gev_result.items():
            if isinstance(value, float):
                print(f"   {key}: {value:.4f}")
            else:
                print(f"   {key}: {value}")
    except Exception as e:
        print(f"⚠️ GEV fit error: {e}")
        print("   (GEV may need more data)")
else:
    print("⚠️ GEV distribution not available.")

In [ ]:
# ============================================================================
# 5. Saving in Different Formats
# ============================================================================

output_dir = os.path.join(project_root, 'advanced_output')
os.makedirs(output_dir, exist_ok=True)

print("💾 Saving results in different formats:")
print("=" * 60)

zarr_path = os.path.join(output_dir, 'results.zarr')
ds_results = xr.Dataset(
    data_vars={
        'mean': (('lat', 'lon'), np.full((len(lat), len(lon)), np.mean(point_data))),
        'std': (('lat', 'lon'), np.full((len(lat), len(lon)), np.std(point_data))),
        'max': (('lat', 'lon'), np.full((len(lat), len(lon)), np.max(point_data))),
        'min': (('lat', 'lon'), np.full((len(lat), len(lon)), np.min(point_data))),
    },
    coords={'lat': lat, 'lon': lon}
)
ds_results.to_zarr(zarr_path, mode='w', consolidated=False)
print(f"✅ Zarr: {zarr_path}")

netcdf_out = os.path.join(output_dir, 'results.nc')
ds_results.to_netcdf(netcdf_out)
print(f"✅ NetCDF: {netcdf_out}")

csv_path = os.path.join(output_dir, 'results.csv')
df_csv = pd.DataFrame({
    'lat': lat,
    'lon': lon,
    'mean': np.full(len(lat), np.mean(point_data)),
    'std': np.full(len(lat), np.std(point_data)),
    'max': np.full(len(lat), np.max(point_data)),
    'min': np.full(len(lat), np.min(point_data))
})
df_csv.to_csv(csv_path, index=False)
print(f"✅ CSV: {csv_path}")

In [ ]:
# ============================================================================
# 6. Analyzing Zarr Output with xarray
# ============================================================================

print("📊 Analyzing Zarr output:")
print("=" * 60)

ds_zarr = xr.open_zarr(zarr_path, consolidated=False)
print(f"   Dimensions: {ds_zarr.dims}")
print(f"   Variables: {list(ds_zarr.data_vars)}")
print(f"   Coordinates: {list(ds_zarr.coords)}")

spatial_mean = ds_zarr['mean'].mean().values
print(f"   Spatial mean: {spatial_mean:.2f}°C")

ds_zarr.close()

In [ ]:
# ============================================================================
# 7. Advanced Maps
# ============================================================================

print("🗺️ Advanced mapping:")

map_data = np.zeros((len(lat), len(lon)))
for i in range(len(lat)):
    for j in range(len(lon)):
        map_data[i, j] = 15 + 5 * np.sin(lat[i] * 0.1) * np.cos(lon[j] * 0.05) + np.random.normal(0, 1)

fig, ax = plt.subplots(figsize=(12, 8))

im = ax.contourf(lon, lat, map_data, 20, cmap='RdBu_r')
cbar = plt.colorbar(im, ax=ax, label='Temperature (°C)')

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('Spatial Mean Temperature Map', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Map with Cartopy (if available)
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    
    fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': ccrs.PlateCarree()})
    
    ax.add_feature(cfeature.LAND, facecolor='lightgray')
    ax.add_feature(cfeature.OCEAN, facecolor='lightblue')
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5)
    ax.add_feature(cfeature.BORDERS, linewidth=0.5, linestyle=':')
    
    im = ax.contourf(lon, lat, map_data, 20, cmap='RdBu_r', 
                     transform=ccrs.PlateCarree())
    cbar = plt.colorbar(im, ax=ax, label='Temperature (°C)', shrink=0.7)
    
    ax.set_extent([44, 64, 25, 40], crs=ccrs.PlateCarree())
    ax.set_title('Spatial Map with Cartopy', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("⚠️ Cartopy not available.")

In [ ]:
# ============================================================================
# 8. Memory Management and block_size
# ============================================================================

print("💾 Memory management:")
print("=" * 60)

def process_in_blocks(data, block_size=100):
    n_points = data.shape[0]
    results = []
    
    for start in range(0, n_points, block_size):
        end = min(start + block_size, n_points)
        block = data[start:end]
        block_result = {
            'start': start,
            'end': end,
            'mean': np.mean(block),
            'std': np.std(block),
            'size': len(block)
        }
        results.append(block_result)
        print(f"   Block {start//block_size + 1}: points {start}-{end}, "
              f"mean = {block_result['mean']:.2f}")
    
    return results

large_data = np.random.randn(1000, 10)
print(f"📊 Large data: {large_data.shape}")

results = process_in_blocks(large_data, block_size=100)
print(f"\n✅ Number of blocks: {len(results)}")

In [ ]:
# ============================================================================
# 9. Using Checkpoint
# ============================================================================

try:
    from monitoring.checkpoint import save_checkpoint, load_checkpoint
except ImportError:
    print("⚠️ checkpoint module not found.")
    checkpoint_dir = os.path.join(output_dir, 'checkpoints')
    os.makedirs(checkpoint_dir, exist_ok=True)
    
    # Simulate checkpoint functions
    def save_checkpoint(dir_path, block, station):
        with open(os.path.join(dir_path, 'checkpoint.txt'), 'w') as f:
            f.write(f"block={block}\nstation={station}\nversion=1\n")
    
    def load_checkpoint(dir_path):
        path = os.path.join(dir_path, 'checkpoint.txt')
        if os.path.exists(path):
            with open(path, 'r') as f:
                lines = f.readlines()
            cp = {}
            for line in lines:
                if '=' in line:
                    k, v = line.strip().split('=', 1)
                    cp[k] = v
            return cp
        return None

print("💾 Checkpoint system:")
print("=" * 60)

checkpoint_dir = os.path.join(output_dir, 'checkpoints')
os.makedirs(checkpoint_dir, exist_ok=True)

save_checkpoint(checkpoint_dir, block=10, station=500)
print(f"✅ Checkpoint saved: block=10, station=500")

cp = load_checkpoint(checkpoint_dir)
if cp:
    print(f"📋 Checkpoint loaded:")
    print(f"   block: {cp.get('block', 'N/A')}")
    print(f"   station: {cp.get('station', 'N/A')}")

In [ ]:
# ============================================================================
# 10. Performance Optimization
# ============================================================================

print("⚡ Performance optimization:")
print("=" * 60)

try:
    from numba import njit
    
    @njit
    def fast_calculation(data):
        return np.mean(data), np.std(data)
    
    data_test = np.random.randn(1000000)
    
    import time
    start = time.time()
    mean, std = fast_calculation(data_test)
    numba_time = time.time() - start
    print(f"✅ With Numba: {numba_time:.4f} seconds")
    
    start = time.time()
    mean, std = np.mean(data_test), np.std(data_test)
    numpy_time = time.time() - start
    print(f"   Without Numba (NumPy): {numpy_time:.4f} seconds")
    print(f"   Speed: {numpy_time/numba_time:.1f}x faster")
    
except ImportError:
    print("⚠️ Numba not installed. To install: pip install numba")

In [ ]:
# ============================================================================
# 11. Summary and Final Notes
# ============================================================================

print("📋 Advanced Usage Summary:")
print("=" * 60)
print("""
✅ Gridded Data support
✅ Extreme Value Mode with GEV
✅ Advanced config.yaml settings
✅ Checkpoint for resuming processing
✅ Memory management and block_size
✅ Multiple output formats (Zarr, NetCDF, CSV)
✅ xarray analysis
✅ Advanced mapping with Cartopy
✅ Numba performance optimization
""")

print("=" * 60)
print("🎉 Advanced Usage complete!")
print(f"📁 Outputs in: {output_dir}")
print("=" * 60)